In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

In [2]:
df = pd.read_csv(
    "opi_builder_c_live_record.csv"
)

print(df.shape)

print()

print(df["Label"].value_counts())

(7013, 12)

Label
NORMAL    6436
ATTACK     577
Name: count, dtype: int64


In [3]:
df["Label"] = df["Label"].map({

    "NORMAL":0,
    "ATTACK":1

})

In [4]:
print(df["Label"].unique())

[0 1]


In [5]:
train_files = [

    "normal_browsing.csv",
    "normal_download.csv",
    "normal_gaming.csv",
    "normal_streaming.csv",

    "syn_flood_basic.csv",
    "syn_u10_basic.csv",
    "syn_u100_basic.csv"

]

test_files = [

    "normal_mixed.csv",

    "syn_flood_randip.csv",
    "syn_randip_basic.csv"

]

In [6]:
train_df = df[
    df["Source_File"].isin(train_files)
]

test_df = df[
    df["Source_File"].isin(test_files)
]

In [7]:
print("TRAIN")

print(train_df.shape)

print()

print(
    train_df["Label"].value_counts()
)

print()

print("="*60)

print()

print("TEST")

print(test_df.shape)

print()

print(
    test_df["Label"].value_counts()
)

TRAIN
(5489, 12)

Label
0    5152
1     337
Name: count, dtype: int64


TEST
(1524, 12)

Label
0    1284
1     240
Name: count, dtype: int64


In [8]:
FEATURES = [

    "Flow Duration",

    "Flow Bytes/s",

    "Flow Packets/s",

    "Total Fwd Packets",

    "Total Length of Fwd Packets",

    "Fwd Packets/s",

    "SYN Flag Count",

    "ACK Flag Count",

    "Init_Win_bytes_forward",

    "act_data_pkt_fwd"

]

In [9]:
X_train = train_df[FEATURES]
y_train = train_df["Label"]

X_test = test_df[FEATURES]
y_test = test_df["Label"]

print(X_train.shape)
print(X_test.shape)

(5489, 10)
(1524, 10)


In [10]:
rf = RandomForestClassifier(

    n_estimators=200,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1

)

rf.fit(
    X_train,
    y_train
)

print("Training Finished")

Training Finished


In [11]:
y_pred = rf.predict(X_test)

In [12]:
acc = accuracy_score(
    y_test,
    y_pred
)

print()

print(
    f"Accuracy = {acc:.4f}"
)

print()

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

print()

print(

    classification_report(

        y_test,
        y_pred,

        target_names=[
            "NORMAL",
            "ATTACK"
        ]

    )

)


Accuracy = 0.9278

[[1284    0]
 [ 110  130]]

              precision    recall  f1-score   support

      NORMAL       0.92      1.00      0.96      1284
      ATTACK       1.00      0.54      0.70       240

    accuracy                           0.93      1524
   macro avg       0.96      0.77      0.83      1524
weighted avg       0.93      0.93      0.92      1524



In [13]:
importance = pd.DataFrame({

    "Feature":FEATURES,

    "Importance":rf.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

importance

,Feature,Importance
6,SYN Flag Count,0.242794
7,ACK Flag Count,0.214645
9,act_data_pkt_fwd,0.198069
8,Init_Win_bytes_forward,0.109553
3,Total Fwd Packets,0.072237
0,Flow Duration,0.058924
4,Total Length of Fwd Packets,0.045258
1,Flow Bytes/s,0.040133
2,Flow Packets/s,0.009756
5,Fwd Packets/s,0.008631


In [14]:
joblib.dump(

    rf,

    "rf_opi_file_split.joblib"

)

print(
    "Saved : rf_opi_file_split.joblib"
)

Saved : rf_opi_file_split.joblib
